# Conversational Hindi & Hinglish NLU: Script-Adaptive Ensemble
### `l3cube-pune/hing-roberta-mixed` (`models/best`) + Calibrated `LinearSVC` Geometric Ensemble

This notebook audits the **<= 300M** parameter budget, trains and cross-validates the Calibrated `LinearSVC` on `data/train.csv` (5-fold Stratified CV), and demonstrates script-adaptive geometric fusion with `models/best`.

In [ ]:
import os, re, json, time, unicodedata
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score, accuracy_score

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Compute Device: {device}')

In [ ]:
# Parameter Budget Audit (<= 300M)
transformer_path = 'models/best'
xlm_tokenizer = AutoTokenizer.from_pretrained(transformer_path)
xlm_model = AutoModelForSequenceClassification.from_pretrained(transformer_path).to(device).eval()
xlm_params = sum(p.numel() for p in xlm_model.parameters())
print(f'XLM-RoBERTa (models/best): {xlm_params:,d} params ({xlm_params/1e6:.2f}M)')

In [ ]:
# Load training data & transliteration dictionary
train_df = pd.read_csv('data/train.csv')
with open('data/xlit_word_dict.json', 'r', encoding='utf-8') as f:
    xlit_dict = json.load(f)

def detect_script(text):
    has_dev = bool(re.search(r'[ऀ-ॿ]', str(text)))
    has_lat = bool(re.search(r'[a-zA-Z]', str(text)))
    if has_dev and has_lat: return 'mixed'
    if has_dev: return 'devanagari'
    if has_lat: return 'latin'
    return 'other'

dev_pattern = re.compile(r'[ऀ-ॿ]+')
asr_map = {
    r'kia': 'kya', r'mjhe': 'mujhe', r'or': 'aur',
    r'h': 'hai', r'mai': 'main', r'plz': 'please',
    r'kr': 'karo', r'thnx': 'dhanyavad', r'chahie': 'chahiye',
    r'btaye': 'batao', r'btado': 'batao', r'soooong': 'song'
}

def clean_asr(text):
    t = unicodedata.normalize('NFKC', str(text)).lower()
    t = re.sub(r'([a-zA-Z]){2,}', r'', t)
    for pat, rep in asr_map.items():
        t = re.sub(pat, rep, t)
    return re.sub(r'\s+', ' ', t).strip()

def transliterate(text):
    t_clean = clean_asr(text)
    if not dev_pattern.search(t_clean):
        return t_clean
    t = dev_pattern.sub(lambda m: xlit_dict.get(m.group(0), m.group(0)), t_clean)
    return re.sub(r'([a-z]){2,}', r'', t.lower())

train_df['script'] = train_df['query_text'].apply(detect_script)
train_df['xlit'] = train_df['query_text'].apply(transliterate)
print('Train script distribution (%):')
print((train_df['script'].value_counts(normalize=True) * 100).round(2))

In [ ]:
# 5-Fold Stratified CV for Latin-Weighted Calibrated LinearSVC on data/train.csv
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train_df), dtype=int)
fold_f1s = []
y_all = train_df['target'].values

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df['xlit'], y_all), 1):
    vec = TfidfVectorizer(ngram_range=(1, 5), analyzer='char_wb', min_df=2, sublinear_tf=True)
    x_tr = vec.fit_transform(train_df['xlit'].iloc[tr_idx])
    x_va = vec.transform(train_df['xlit'].iloc[va_idx])
    sw = np.ones(len(tr_idx), dtype=np.float32)
    sw[(train_df['script'].iloc[tr_idx] == 'latin').values] = 2.0
    clf = CalibratedClassifierCV(LinearSVC(C=1.0, random_state=42, max_iter=2000), cv=3)
    clf.fit(x_tr, y_all[tr_idx], sample_weight=sw)
    preds = clf.predict(x_va)
    oof_preds[va_idx] = preds
    f1 = f1_score(y_all[va_idx], preds, average='macro')
    fold_f1s.append(f1)
    print(f'Fold {fold} Macro-F1: {f1:.4f}')

print(f'Mean 5-Fold Macro-F1: {np.mean(fold_f1s):.4f} | OOF Macro-F1: {f1_score(y_all, oof_preds, average="macro"):.4f}')

In [ ]:
# Fit full-train Calibrated LinearSVC & verify exact parameter budget
tfidf_vec = TfidfVectorizer(ngram_range=(1, 5), analyzer='char_wb', min_df=2, sublinear_tf=True)
X_train_vec = tfidf_vec.fit_transform(train_df['xlit'])
sample_weights = np.ones(len(train_df), dtype=np.float32)
sample_weights[(train_df['script'] == 'latin').values] = 2.0

calibrated_svc = CalibratedClassifierCV(LinearSVC(C=1.0, random_state=42, max_iter=2000), cv=3)
calibrated_svc.fit(X_train_vec, y_all, sample_weight=sample_weights)

svc_params = len(tfidf_vec.vocabulary_) * 56 + 56
total_active = xlm_params + svc_params
print(f'LinearSVC params: {svc_params:,d} | Total Active Params: {total_active:,d} ({total_active/1e6:.2f}M / 300.00M)')
assert total_active <= 300_000_000

In [ ]:
# Script-Adaptive Geometric Inference Helper (live single-query inference)
SCRIPT_WEIGHTS = {'latin': 0.52, 'devanagari': 0.32, 'mixed': 0.45, 'other': 0.45}

def predict_query(query_text, oos_conf=0.50):
    script = detect_script(query_text)
    inputs = xlm_tokenizer([unicodedata.normalize('NFKC', query_text).strip()], return_tensors='pt', truncation=True, max_length=128).to(device)
    with torch.no_grad():
        p_xlm = torch.softmax(xlm_model(**inputs).logits, dim=-1)[0].cpu().numpy()
    p_svc = calibrated_svc.predict_proba(tfidf_vec.transform([transliterate(query_text)]))[0]
    g = SCRIPT_WEIGHTS[script]
    p_ens = np.exp(g * np.log(p_xlm + 0.01) + (1.0 - g) * np.log(p_svc + 0.01))
    p_ens /= p_ens.sum()
    p_ens[55] *= 1.30
    p_ens[53] *= 1.15
    p_ens[12] *= 1.15
    p_ens /= p_ens.sum()
    pred = int(np.argmax(p_ens))
    if pred == 55 and p_ens[pred] < oos_conf:
        masked = p_ens.copy(); masked[55] = -1
        pred = int(np.argmax(masked))
    return pred, float(p_ens[pred]), script

for q in ['kripaya rasoeeghar ki battiyaan band kar den', 'शनिवार का पूर्वानुमान क्या है', 'wemo plug socket बंद करें']:
    print(q, '->', predict_query(q))